In [10]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('dog-breed-identification')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/dog-breed-identification


In [4]:
import torch
import os
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import torch.optim as optim
import torchvision.models as models
import torch.nn.functional as f
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from sklearn.model_selection import StratifiedKFold

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

GPU: Tesla T4


In [12]:
df = pd.read_csv('/kaggle/input/competitions/dog-breed-identification/labels.csv')
is_unique = df['breed'].is_unique
print(is_unique)

False


In [13]:
unique_breeds = df['breed'].unique()
breed_to_index = {breed: index for index, breed in enumerate(unique_breeds)}
df['label'] = df['breed'].map(breed_to_index)

In [14]:
df

,id,breed,label
0,000bec180eb18c7604dcecc8fe0dba07,boston_bull,0
1,001513dfcb2ffafc82cccf4d8bbaba97,dingo,1
2,001cdf01b096e06d78e9e5112d419397,pekinese,2
3,00214f311d5d2247d5dfe4fe24b2303d,bluetick,3
4,0021f9ceb3235effd7fcde7f7538ed62,golden_retriever,4
...,...,...,...
10217,ffd25009d635cfd16e793503ac5edef0,borzoi,6
10218,ffd3f636f7f379c51ba3648a9ff8254f,dandie_dinmont,93
10219,ffe2ca6c940cddfee68fa3cc6c63213f,airedale,63
10220,ffe5f6d8e2bff356e9482a80a6e29aac,miniature_pinscher,77


In [15]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [16]:
class DogBreedDataset(Dataset):
    def __init__(self, dataframe, image_directory, transform=None):
        self.dataframe = dataframe
        self.image_directory = image_directory
        self.transform = transform

    
    def __len__(self):
        return len(self.dataframe)


    def __getitem__(self, index):
        image_name = self.dataframe.iloc[index]['id'] + '.jpg'
        image_path = os.path.join(self.image_directory, image_name)
        image = Image.open(image_path).convert('RGB')
        label = self.dataframe.iloc[index]['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [17]:
class CNN(nn.Module):
    def __init__(self, num_classes=120):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 64 * 64, 256),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(256, num_classes)
        )

    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)

        return x

In [18]:
dummy_images = torch.randn(2, 3, 256, 256)
model = CNN()
output = model.conv_layers(dummy_images)

print(output.shape)

torch.Size([2, 256, 64, 64])


In [21]:
image_dir = '/kaggle/input/competitions/dog-breed-identification/train'
stratified_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, validation_index) in enumerate(stratified_fold.split(X=df['id'], y=df['label'])):
    train_df = df.iloc[train_index].reset_index(drop=True)
    validation_df = df.iloc[validation_index].reset_index(drop=True)

    train_dataset = DogBreedDataset(dataframe=train_df, image_directory=image_dir, transform=transform)
    validation_dataset = DogBreedDataset(dataframe=validation_df, image_directory=image_dir, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, num_workers=2)

    model = CNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    num_epochs = 5
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs= model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        epoch_train_loss = train_loss / train_total
        epoch_train_accuracy = (train_correct / train_total) * 100

        model.eval()
        validation_loss = 0.0
        validation_correct = 0
        validation_total = 0

        with torch.no_grad():
            for images, labels in validation_loader:
                images, labels = images.to(device), labels.to(device)
                outputs= model(images)
                loss = criterion(outputs, labels)
    
                validation_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                validation_total += labels.size(0)
                validation_correct += (predicted == labels).sum().item()
    
            epoch_validation_loss = validation_loss / validation_total
            epoch_validation_accuracy = (validation_correct / validation_total) * 100

        print(f'Epoch [{epoch + 1} / {num_epochs}] ' 
            f'Train Loss: {epoch_train_loss:.4f}, Train Accuracy: {epoch_train_accuracy:.2f}% | '
            f'Validation Loss: {epoch_validation_loss:.4f}, Validation Accuracy: {epoch_validation_accuracy:.2f}%')

Epoch [1 / 5] Train Loss: 13.6550, Train Accuracy: 0.88% | Validation Loss: 4.7864, Validation Accuracy: 1.17%
Epoch [2 / 5] Train Loss: 4.7856, Train Accuracy: 1.10% | Validation Loss: 4.7835, Validation Accuracy: 1.12%
Epoch [3 / 5] Train Loss: 4.7835, Train Accuracy: 1.04% | Validation Loss: 4.7819, Validation Accuracy: 1.12%
Epoch [4 / 5] Train Loss: 4.7819, Train Accuracy: 1.09% | Validation Loss: 4.7805, Validation Accuracy: 1.12%
Epoch [5 / 5] Train Loss: 4.7807, Train Accuracy: 1.10% | Validation Loss: 4.7794, Validation Accuracy: 1.12%
Epoch [1 / 5] Train Loss: 13.7645, Train Accuracy: 0.76% | Validation Loss: 4.7857, Validation Accuracy: 1.12%
Epoch [2 / 5] Train Loss: 4.7866, Train Accuracy: 0.93% | Validation Loss: 4.7835, Validation Accuracy: 1.12%
Epoch [3 / 5] Train Loss: 4.7834, Train Accuracy: 1.00% | Validation Loss: 4.7817, Validation Accuracy: 1.12%
Epoch [4 / 5] Train Loss: 4.7819, Train Accuracy: 1.11% | Validation Loss: 4.7804, Validation Accuracy: 1.12%
Epoch [5

In [26]:
class CNN_second(nn.Module):
    def __init__(self, num_classes=120):
        super(CNN_second, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.5),
            nn.Linear(512, num_classes)
        )

    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.global_pool(x)
        x = self.fc_layers(x)

        return x

In [27]:
dummy_images = torch.randn(2, 3, 256, 256)
model = CNN_second()
output = model.conv_layers(dummy_images)

print(output.shape)

torch.Size([2, 512, 16, 16])


In [28]:
image_dir = '/kaggle/input/competitions/dog-breed-identification/train'
stratified_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, validation_index) in enumerate(stratified_fold.split(X=df['id'], y=df['label'])):
    train_df = df.iloc[train_index].reset_index(drop=True)
    validation_df = df.iloc[validation_index].reset_index(drop=True)

    train_dataset = DogBreedDataset(dataframe=train_df, image_directory=image_dir, transform=transform)
    validation_dataset = DogBreedDataset(dataframe=validation_df, image_directory=image_dir, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, num_workers=2)

    model = CNN_second().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    num_epochs = 5
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs= model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        epoch_train_loss = train_loss / train_total
        epoch_train_accuracy = (train_correct / train_total) * 100

        model.eval()
        validation_loss = 0.0
        validation_correct = 0
        validation_total = 0

        with torch.no_grad():
            for images, labels in validation_loader:
                images, labels = images.to(device), labels.to(device)
                outputs= model(images)
                loss = criterion(outputs, labels)
    
                validation_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                validation_total += labels.size(0)
                validation_correct += (predicted == labels).sum().item()
    
            epoch_validation_loss = validation_loss / validation_total
            epoch_validation_accuracy = (validation_correct / validation_total) * 100

        print(f'Epoch [{epoch + 1} / {num_epochs}] ' 
            f'Train Loss: {epoch_train_loss:.4f}, Train Accuracy: {epoch_train_accuracy:.2f}% | '
            f'Validation Loss: {epoch_validation_loss:.4f}, Validation Accuracy: {epoch_validation_accuracy:.2f}%')

Epoch [1 / 5] Train Loss: 4.8569, Train Accuracy: 1.49% | Validation Loss: 4.7118, Validation Accuracy: 2.15%
Epoch [2 / 5] Train Loss: 4.7138, Train Accuracy: 2.20% | Validation Loss: 4.6552, Validation Accuracy: 1.91%
Epoch [3 / 5] Train Loss: 4.6375, Train Accuracy: 2.52% | Validation Loss: 4.6100, Validation Accuracy: 2.15%
Epoch [4 / 5] Train Loss: 4.5922, Train Accuracy: 3.25% | Validation Loss: 4.5717, Validation Accuracy: 2.93%
Epoch [5 / 5] Train Loss: 4.5561, Train Accuracy: 4.12% | Validation Loss: 4.5512, Validation Accuracy: 3.72%
Epoch [1 / 5] Train Loss: 4.8429, Train Accuracy: 1.31% | Validation Loss: 4.7107, Validation Accuracy: 2.00%
Epoch [2 / 5] Train Loss: 4.7228, Train Accuracy: 1.96% | Validation Loss: 4.6519, Validation Accuracy: 2.40%
Epoch [3 / 5] Train Loss: 4.6541, Train Accuracy: 2.34% | Validation Loss: 4.5941, Validation Accuracy: 3.08%
Epoch [4 / 5] Train Loss: 4.6079, Train Accuracy: 3.17% | Validation Loss: 4.5669, Validation Accuracy: 3.62%
Epoch [5 /

In [36]:
def get_model(model_name, num_classes=120):
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for parameter in model.parameters():
            parameter.requires_grad = False

        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
        optimizer_params = model.fc.parameters()

    elif model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for parameter in model.parameters():
            parameter.requires_grad = False

        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
        optimizer_params = model.fc.parameters()

    elif model_name == 'mobilenet_v2':
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        for parameter in model.parameters():
            parameter.requires_grad = False

        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
        optimizer_params = model.classifier[1].parameters()

    return model, optimizer_params

In [37]:
models_for_tests = ['resnet18', 'resnet50', 'mobilenet_v2']
best_accuracy = 0.0
best_model_name = ''

for architecture in models_for_tests:
    print(f'\n' + '='*40)
    print(f"   STARTING BENCHMARK: {architecture.upper()}")
    print("="*40)

    for fold, (train_index, validation_index) in enumerate(stratified_fold.split(X=df['id'], y=df['label'])):
        train_df = df.iloc[train_index].reset_index(drop=True)
        validation_df = df.iloc[validation_index].reset_index(drop=True)
    
        train_dataset = DogBreedDataset(dataframe=train_df, image_directory=image_dir, transform=transform)
        validation_dataset = DogBreedDataset(dataframe=validation_df, image_directory=image_dir, transform=transform)
    
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
        validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, num_workers=2)

        model, optimizer_parameters = get_model(architecture, num_classes=120)
        model = model.to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(optimizer_parameters, lr=0.001)
        
        num_epochs = 10
        for epoch in range(num_epochs):
            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0
    
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs= model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
    
                train_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                train_total += labels.size(0)
                train_correct += (predicted == labels).sum().item()
    
            epoch_train_loss = train_loss / train_total
            epoch_train_accuracy = (train_correct / train_total) * 100
    
            model.eval()
            validation_loss = 0.0
            validation_correct = 0
            validation_total = 0
    
            with torch.no_grad():
                for images, labels in validation_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs= model(images)
                    loss = criterion(outputs, labels)
        
                    validation_loss += loss.item() * images.size(0)
                    _, predicted = torch.max(outputs, 1)
                    validation_total += labels.size(0)
                    validation_correct += (predicted == labels).sum().item()
        
                epoch_validation_loss = validation_loss / validation_total
                epoch_validation_accuracy = (validation_correct / validation_total) * 100
    
            print(f'Epoch [{epoch + 1} / {num_epochs}] ' 
                f'Train Loss: {epoch_train_loss:.4f}, Train Accuracy: {epoch_train_accuracy:.2f}% | '
                f'Validation Loss: {epoch_validation_loss:.4f}, Validation Accuracy: {epoch_validation_accuracy:.2f}%')

            if epoch_validation_accuracy > best_accuracy:
                best_accuracy = epoch_validation_accuracy
                best_model_name = architecture
                torch.save(model.state_dict(), 'best_model.pth')

                print(f'Saved {architecture} with {best_accuracy:.2f}%')


   STARTING BENCHMARK: RESNET18
Epoch [1 / 10] Train Loss: 2.8556, Train Accuracy: 39.65% | Validation Loss: 1.5927, Validation Accuracy: 63.72%
Saved resnet18 with 63.72%
Epoch [2 / 10] Train Loss: 1.2844, Train Accuracy: 70.22% | Validation Loss: 1.1322, Validation Accuracy: 69.78%
Saved resnet18 with 69.78%
Epoch [3 / 10] Train Loss: 0.9219, Train Accuracy: 77.09% | Validation Loss: 0.9723, Validation Accuracy: 72.27%
Saved resnet18 with 72.27%
Epoch [4 / 10] Train Loss: 0.7554, Train Accuracy: 80.56% | Validation Loss: 0.9004, Validation Accuracy: 74.38%
Saved resnet18 with 74.38%
Epoch [5 / 10] Train Loss: 0.6425, Train Accuracy: 83.54% | Validation Loss: 0.8654, Validation Accuracy: 74.47%
Saved resnet18 with 74.47%
Epoch [6 / 10] Train Loss: 0.5717, Train Accuracy: 85.07% | Validation Loss: 0.8251, Validation Accuracy: 75.99%
Saved resnet18 with 75.99%
Epoch [7 / 10] Train Loss: 0.4972, Train Accuracy: 87.23% | Validation Loss: 0.8590, Validation Accuracy: 73.40%
Epoch [8 / 10]

In [38]:
class TestDataset(Dataset):
    def __init__(self, test_directory, transform=None):
        self.test_directory = test_directory
        self.image_names = [file for file in os.listdir(test_directory) if file.endswith('.jpg')]
        self.transform = transform


    def __len__(self):
        return len(self.image_names)


    def __getitem__(self, index):
        image_name = self.image_names[index]
        image_path = os.path.join(self.test_directory, image_name)
        image = Image.open(image_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        image_id = image_name.replace('.jpg', '')
        return image, image_id

In [39]:
test_directory = '/kaggle/input/competitions/dog-breed-identification/test'
test_dataset = TestDataset(test_directory=test_directory, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [40]:
all_probabilities = []
all_image_ids = []

final_model, _ = get_model(best_model_name, num_classes=120)
final_model.load_state_dict(torch.load('best_model.pth'))
final_model = final_model.to(device)
model.eval()

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        probabilities = f.softmax(outputs, dim=1)

        all_probabilities.extend(probabilities.cpu().numpy())
        all_image_ids.extend(image_ids)

In [41]:
index_to_breed = {index: breed for breed, index in breed_to_index.items()}
breed_columns = [index_to_breed[i] for i in range(120)]

submission = pd.DataFrame(all_probabilities, columns=breed_columns)
submission.insert(0, 'id', all_image_ids)
submission.to_csv('submission.csv', index=False)